# CHARM oracle rollout comparison

This notebook runs CHARM on one `SIM_ID` and compares free-running inference against oracle rollouts that replace specific upstream sampled quantities with truth. It is designed to answer whether catalog statistic errors are mainly coming from binary/count sampling, mass sampling, or downstream position/property sampling.

Rollouts produced here:

- `truth_all`: reconstructed training-format truth targets.
- `free`: normal inference, all heads sampled.
- `true_count_sampled_downstream`: truth binary/count, sampled masses/positions/velocities/concentrations.
- `true_count_truth_mass_sampled_props`: truth binary/count and truth masses, sampled positions/velocities/concentrations.
- `sampled_count_truth_overlap`: sampled count; truth masses/properties for slots that exist in both truth and sampled catalogs, sampled values kept for extra false-positive slots.

Set `CONFIG_PATH`, `SIM_ID`, and optionally `CHECKPOINT_PATH`, then run top to bottom.

In [ ]:
from pathlib import Path
import os
import sys
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

REPO_ROOT = Path("/mnt/ceph/users/spandey/CHARM_v2/CHARM").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "charm") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "charm"))

from charm.config_loader import load_config
from charm.run_charm_joint_ddp import build_model
from charm.run_inference_v2 import (
    build_cond_tensors,
    build_dm_velocity_interpolators,
    estimate_target_prior_from_cosmology,
    load_checkpoint,
    load_cosmology,
    load_fastpm,
    subvols_unpadded,
)
from charm.utils_data_prep_v2 import prep_halo_catalog

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.grid": True,
    "grid.alpha": 0.18,
})

In [ ]:
# User parameters.
CONFIG_PATH = REPO_ROOT / "run_configs/TRAIN_CHARM_JOINT_v2vel.yaml"
SIM_ID = 1903

# Leave as None to use the best checkpoint under train_settings.checkpoint_dir.
CHECKPOINT_PATH = None

# Binary prior controls. These mirror charm/run_inference_v2.py.
BINARY_TARGET_PRIOR = None
BINARY_PRIOR_CALIBRATOR = None
BINARY_TRAIN_PRIOR = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 12345

# Power-spectrum settings for each 125 Mpc/h subvolume.
RUN_POWER_SPECTRA = True
MAX_POWER_SUBVOLS = None  # set to e.g. 64 for a quick pass
POWER_GRID = 64
K_MAX = 1.0
MAS = "TSC"
RSD_AXIS = 2
RATIO_YLIM = (0.5, 1.5)

FIG_PATH = REPO_ROOT / "notebooks/testing/oracle_rollout_power_ratios.png"

In [ ]:
def resolve_repo_path(path_like):
    if path_like is None:
        return None
    p = Path(path_like)
    return p if p.is_absolute() else (REPO_ROOT / p).resolve()


def resolve_best_checkpoint(cfg, explicit_path=None):
    if explicit_path is not None:
        return resolve_repo_path(explicit_path)

    ckpt_dir = resolve_repo_path(cfg["train_settings"]["checkpoint_dir"])
    candidates = [
        ckpt_dir / "checkpoint_best_rollout.pth",
        ckpt_dir / "charm_joint_best_val.pth",
        ckpt_dir / "checkpoint_best_teacher.pth",
        ckpt_dir / "best_val.pth",
        ckpt_dir / "checkpoint_best.pth",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    best_like = sorted(ckpt_dir.glob("*best*.pth"))
    if best_like:
        return best_like[-1]
    raise FileNotFoundError(
        f"No best checkpoint found in {ckpt_dir}. Set CHECKPOINT_PATH explicitly."
    )


cfg = load_config(str(resolve_repo_path(CONFIG_PATH)))
sc = cfg["sim_settings"]
dc = cfg["data_settings"]
tc = cfg["train_settings"]

nb = int(sc["nb"])
ns_d = int(sc["ns_d"])
ns_h = int(sc["ns_h"])
nax = ns_h // nb
nsubs = nb ** 3
nvox = nax ** 3
total_vox = nsubs * nvox
Nmax = int(sc["Nmax"])
BoxSize = float(dc["BoxSize"])
Lsub = BoxSize / nb
z_snap = str(dc["z_snap"])

n_cnn_tot = sum(1 if lt == "cnn" else 2 for lt in sc["layers_types"])
n_pad = (int(sc["nf"]) - 1) // 2 * n_cnn_tot

z_by_snap = {4: 0.0, 3: 0.5, 2: 1.0, 1: 2.0, 0: 3.0, -1: 99.0}
redshift = z_by_snap[int(dc["snapnum"])]

lgMmin = float(sc["lgMmin"])
lgMmax = float(sc["lgMmax"])
vmin = float(sc["vmin"])
vmax = float(sc["vmax"])
cmin = float(sc["cmin"])
cmax = float(sc["cmax"])

ckpt_path = resolve_best_checkpoint(cfg, CHECKPOINT_PATH)

assert nb == 8 and nsubs == 512, f"Expected 8^3=512 subvolumes; got nb={nb}."
assert ns_d == ns_h, f"This notebook assumes matching DM and halo grids; got {ns_d} and {ns_h}."

print(f"config: {resolve_repo_path(CONFIG_PATH)}")
print(f"simulation id: {SIM_ID}")
print(f"checkpoint: {ckpt_path}")
print(f"subvolumes: {nsubs} x ({nax}^3 voxels), Lsub={Lsub:.3f} Mpc/h")
print(f"device: {DEVICE}")

In [ ]:
def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def flat_subvolume_voxel_indices(nb, nax):
    nsubs = nb ** 3
    nvox = nax ** 3
    total = nsubs * nvox

    isub = np.arange(total, dtype=np.int64) // nvox
    ivox = np.arange(total, dtype=np.int64) % nvox

    jx = isub // (nb * nb)
    jy = (isub % (nb * nb)) // nb
    jz = isub % nb

    vx = ivox // (nax * nax)
    vy = (ivox % (nax * nax)) // nax
    vz = ivox % nax
    return isub, jx, jy, jz, vx, vy, vz


def new_catalog_lists(n):
    return [{"pos": [], "lgM": [], "vel": [], "conc": []} for _ in range(n)]


def append_grouped(cats, sub_ids, pos, lgM, vel, conc):
    if len(sub_ids) == 0:
        return
    order = np.argsort(sub_ids, kind="mergesort")
    sid_sorted = sub_ids[order]
    cuts = np.r_[0, np.flatnonzero(sid_sorted[1:] != sid_sorted[:-1]) + 1, len(order)]

    for start, end in zip(cuts[:-1], cuts[1:]):
        sid = int(sid_sorted[start])
        idx = order[start:end]
        cats[sid]["pos"].append(pos[idx].astype(np.float32, copy=False))
        cats[sid]["lgM"].append(lgM[idx].astype(np.float32, copy=False))
        cats[sid]["vel"].append(vel[idx].astype(np.float32, copy=False))
        cats[sid]["conc"].append(conc[idx].astype(np.float32, copy=False))


def finalize_catalogs(cats):
    out = []
    for cat in cats:
        if cat["pos"]:
            out.append({
                "pos": np.concatenate(cat["pos"], axis=0).astype(np.float32),
                "lgM": np.concatenate(cat["lgM"], axis=0).astype(np.float32),
                "vel": np.concatenate(cat["vel"], axis=0).astype(np.float32),
                "conc": np.concatenate(cat["conc"], axis=0).astype(np.float32),
            })
        else:
            out.append({
                "pos": np.zeros((0, 3), dtype=np.float32),
                "lgM": np.zeros(0, dtype=np.float32),
                "vel": np.zeros((0, 3), dtype=np.float32),
                "conc": np.zeros(0, dtype=np.float32),
            })
    return out


def reconstruct_sample_out_by_subvolume(sample_out, vel_interps):
    total = nsubs * nvox
    cell = BoxSize / ns_h
    v_range = vmax - vmin

    ntot = np.clip(np.rint(to_numpy(sample_out["ntot"][0]).reshape(total)), 0, Nmax).astype(np.int16)
    m1 = np.clip(to_numpy(sample_out["m1"][0]).reshape(total), 0.0, 1.0).astype(np.float32)
    mdiff = np.clip(to_numpy(sample_out["mdiff"][0]).reshape(total, Nmax - 1), 0.0, 1.0).astype(np.float32)
    vel_norm = np.clip(to_numpy(sample_out["vel"][0]).reshape(total, Nmax * 3), -0.5, 0.5).astype(np.float32)
    conc_norm = np.clip(to_numpy(sample_out["conc"][0]).reshape(total, Nmax), 0.0, 1.0).astype(np.float32)
    pos_norm = np.clip(to_numpy(sample_out["pos"][0]).reshape(total, Nmax * 3), -0.5, 0.5).astype(np.float32)

    M_norm = np.zeros((total, Nmax), dtype=np.float32)
    M_norm[:, 0] = m1
    for ih in range(1, Nmax):
        M_norm[:, ih] = np.clip(M_norm[:, ih - 1] - mdiff[:, ih - 1], 0.0, 1.0)
    lgM_grid = M_norm * (lgMmax - lgMmin) + lgMmin

    isub, jx, jy, jz, vx, vy, vz = flat_subvolume_voxel_indices(nb, nax)
    cats = new_catalog_lists(nsubs)

    for ih in range(Nmax):
        has_halo = ntot > ih
        if not np.any(has_halo):
            continue

        px = pos_norm[has_halo, ih * 3 + 0]
        py = pos_norm[has_halo, ih * 3 + 1]
        pz = pos_norm[has_halo, ih * 3 + 2]

        x_local = ((vx[has_halo] + px) * cell) % Lsub
        y_local = ((vy[has_halo] + py) * cell) % Lsub
        z_local = ((vz[has_halo] + pz) * cell) % Lsub
        pos_local = np.stack([x_local, y_local, z_local], axis=1).astype(np.float32)

        x_full = (((jx[has_halo] * nax + vx[has_halo]) + px) * cell) % BoxSize
        y_full = (((jy[has_halo] * nax + vy[has_halo]) + py) * cell) % BoxSize
        z_full = (((jz[has_halo] * nax + vz[has_halo]) + pz) * cell) % BoxSize
        pos_full = np.stack([x_full, y_full, z_full], axis=1).astype(np.float32)

        v_dm = np.stack([interp(pos_full) for interp in vel_interps], axis=1).astype(np.float32)
        v_diff = vel_norm[has_halo, ih * 3: ih * 3 + 3] * v_range
        vel_phys = (v_dm - v_diff).astype(np.float32)

        lgM = lgM_grid[has_halo, ih].astype(np.float32)
        conc = (conc_norm[has_halo, ih] * (cmax - cmin) + cmin).astype(np.float32)
        append_grouped(cats, isub[has_halo], pos_local, lgM, vel_phys, conc)

    return finalize_catalogs(cats)


def merge_subvolume_catalogs(cats):
    if not cats:
        return {"pos": np.zeros((0, 3), dtype=np.float32), "lgM": np.zeros(0, dtype=np.float32), "vel": np.zeros((0, 3), dtype=np.float32), "conc": np.zeros(0, dtype=np.float32)}
    out = {}
    for key, shape in [("pos", (0, 3)), ("lgM", (0,)), ("vel", (0, 3)), ("conc", (0,))]:
        pieces = [cat[key] for cat in cats if len(cat[key]) > 0]
        out[key] = np.concatenate(pieces, axis=0).astype(np.float32) if pieces else np.zeros(shape, dtype=np.float32)
    return out

In [ ]:
def halo_mass_function(lgM, lgMmin, lgMmax, n_bins=20, BoxSize=1000.0):
    bins = np.linspace(lgMmin, lgMmax, n_bins + 1)
    counts, _ = np.histogram(lgM, bins=bins)
    dlgM = bins[1] - bins[0]
    volume = BoxSize ** 3
    dn_dlgM = counts / (volume * dlgM)
    return 0.5 * (bins[:-1] + bins[1:]), dn_dlgM


def _density_field(pos, weights, Ng, BoxSize, MAS="TSC"):
    import MAS_library as MASL

    pos32 = np.ascontiguousarray(pos.astype(np.float32))
    delta = np.zeros((Ng, Ng, Ng), dtype=np.float32)
    if weights is None:
        MASL.MA(pos32, delta, np.float32(BoxSize), MAS)
    else:
        w32 = np.ascontiguousarray(weights.astype(np.float32))
        MASL.MA(pos32, delta, np.float32(BoxSize), MAS, W=w32)

    mean = delta.mean(dtype=np.float64)
    if mean <= 0:
        return None
    return (delta / mean - 1.0).astype(np.float32)


def power_spectrum(pos, weights, Ng, BoxSize, kmax=0.4, MAS="TSC", axis=0):
    import Pk_library as PKL

    delta = _density_field(pos, weights, Ng, BoxSize, MAS=MAS)
    if delta is None:
        return np.zeros(1), np.zeros(1)

    pk_obj = PKL.Pk(delta, np.float32(BoxSize), axis=axis, MAS=MAS, threads=1, verbose=False)
    k = pk_obj.k3D
    pk = pk_obj.Pk[:, 0]
    sel = (k > 0) & (k <= kmax) & np.isfinite(pk) & (pk > 0)
    return k[sel], pk[sel]


def apply_rsd(pos, vel, BoxSize, z, cosmo, axis=2):
    omega_m = float(cosmo[0])
    Ez = np.sqrt(omega_m * (1.0 + z) ** 3 + (1.0 - omega_m))
    H_eff = 100.0 * Ez
    pos_rsd = pos.copy()
    pos_rsd[:, axis] = (pos[:, axis] + vel[:, axis] * (1.0 + z) / H_eff) % BoxSize
    return pos_rsd

In [ ]:
def sample_arrays(sample_out):
    total = nsubs * nvox
    ntot = np.clip(np.rint(to_numpy(sample_out["ntot"][0]).reshape(total)), 0, Nmax).astype(np.int16)
    m1 = np.clip(to_numpy(sample_out["m1"][0]).reshape(total), 0.0, 1.0).astype(np.float32)
    mdiff = np.clip(to_numpy(sample_out["mdiff"][0]).reshape(total, Nmax - 1), 0.0, 1.0).astype(np.float32)
    vel = np.clip(to_numpy(sample_out["vel"][0]).reshape(total, Nmax * 3), -0.5, 0.5).astype(np.float32)
    conc = np.clip(to_numpy(sample_out["conc"][0]).reshape(total, Nmax), 0.0, 1.0).astype(np.float32)
    pos = np.clip(to_numpy(sample_out["pos"][0]).reshape(total, Nmax * 3), -0.5, 0.5).astype(np.float32)

    mnorm = np.zeros((total, Nmax), dtype=np.float32)
    mnorm[:, 0] = m1
    for ih in range(1, Nmax):
        mnorm[:, ih] = np.clip(mnorm[:, ih - 1] - mdiff[:, ih - 1], 0.0, 1.0)
    return {"ntot": ntot, "M_norm": mnorm, "vel": vel, "conc": conc, "pos": pos}


def sample_out_from_arrays(arr):
    mnorm = np.clip(arr["M_norm"].astype(np.float32, copy=False), 0.0, 1.0)
    mdiff = np.clip(mnorm[:, :-1] - mnorm[:, 1:], 0.0, 1.0).astype(np.float32)
    return {
        "ntot": [arr["ntot"].astype(np.float32, copy=False)],
        "m1": [mnorm[:, 0].astype(np.float32, copy=False)],
        "mdiff": [mdiff],
        "vel": [arr["vel"].astype(np.float32, copy=False)],
        "conc": [arr["conc"].astype(np.float32, copy=False)],
        "pos": [arr["pos"].astype(np.float32, copy=False)],
    }


def hybrid_sampled_count_truth_overlap(sample_out_free, sample_out_truth):
    """
    Use sampled ntot. For halo slots present in both sampled and truth catalogs,
    replace mass/position/velocity/concentration with truth. For sampled extra
    slots with no truth label, keep sampled downstream values so the catalog is
    still physically decodable.
    """
    free = sample_arrays(sample_out_free)
    truth = sample_arrays(sample_out_truth)
    slot = np.arange(Nmax, dtype=np.int16)[None, :]
    overlap = slot < np.minimum(free["ntot"], truth["ntot"])[:, None]
    overlap3 = np.repeat(overlap, 3, axis=1)

    out = {key: value.copy() for key, value in free.items()}
    out["M_norm"][overlap] = truth["M_norm"][overlap]
    out["conc"][overlap] = truth["conc"][overlap]
    out["pos"][overlap3] = truth["pos"][overlap3]
    out["vel"][overlap3] = truth["vel"][overlap3]
    return sample_out_from_arrays(out)


def count_pdf(ntot, max_n):
    vals = np.clip(ntot.astype(np.int64), 0, max_n)
    return np.bincount(vals, minlength=max_n + 1).astype(np.int64)


def rollout_diagnostics(sample_out):
    arr = sample_arrays(sample_out)
    ntot = arr["ntot"]
    slot = np.arange(Nmax, dtype=np.int16)[None, :]
    valid_pairs = slot[:, :-1] < np.maximum(ntot[:, None] - 1, 0)
    if np.any(valid_pairs):
        mono_bad = np.mean((arr["M_norm"][:, :-1] < arr["M_norm"][:, 1:] - 1e-6)[valid_pairs])
    else:
        mono_bad = 0.0

    numeric = [arr["M_norm"], arr["vel"], arr["conc"], arr["pos"]]
    n_bad = sum(np.size(x) - np.isfinite(x).sum() for x in numeric)
    n_total = sum(np.size(x) for x in numeric)
    return {
        "N_occ": int((ntot > 0).sum()),
        "N_halo": int(ntot.sum()),
        "mean_N_occupied": float(ntot[ntot > 0].mean()) if np.any(ntot > 0) else 0.0,
        "mass_mono_bad_frac": float(mono_bad),
        "nan_inf_frac": float(n_bad / max(n_total, 1)),
        "Ntot_pdf": count_pdf(ntot, Nmax),
    }


def show_table(rows):
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)

In [ ]:
print("Loading FastPM density and velocity fields...")
rho, vel = load_fastpm(dc["fastpm_dir"], SIM_ID, ns_d, z_snap)
cosmo_vals = load_cosmology(dc["lh_cosmo_file"], SIM_ID)
vel_interps = build_dm_velocity_interpolators(vel, BoxSize)

print("Building CHARM conditioning tensors...")
cond_x, cond_x_nsh, cond_cosmo = build_cond_tensors(rho, vel, cosmo_vals, nb, nax, n_pad)
cond_x = cond_x.to(DEVICE)
cond_x_nsh = cond_x_nsh.to(DEVICE)
cond_cosmo = cond_cosmo.to(DEVICE)

print("Loading model and checkpoint...")
model = build_model(cfg).to(DEVICE)
model = load_checkpoint(model, str(ckpt_path), torch.device(DEVICE))
model.eval()

if BINARY_TRAIN_PRIOR is not None:
    model.binary_train_prior = float(BINARY_TRAIN_PRIOR)

binary_target_prior = BINARY_TARGET_PRIOR
if model.binary_train_prior is not None and binary_target_prior is None:
    cal_path = resolve_repo_path(BINARY_PRIOR_CALIBRATOR) if BINARY_PRIOR_CALIBRATOR else ckpt_path.parent / "binary_prior_calibrator.npz"
    if cal_path.exists():
        binary_target_prior, info = estimate_target_prior_from_cosmology(cosmo_vals, str(cal_path))
        print(f"Auto-estimated binary_target_prior={binary_target_prior:.3e} from {cal_path}")
    else:
        warnings.warn(
            "Model has a binary_train_prior but no binary target prior or calibrator was found; using uncorrected occupancy."
        )

print(f"binary_train_prior={model.binary_train_prior}")
print(f"binary_target_prior={binary_target_prior}")

In [ ]:
def load_processed_halo_h5():
    halo_dir = resolve_repo_path(dc["halo_hdf5_dir"])
    fpath = halo_dir / str(SIM_ID) / f"halos_rockstar_200c_z{redshift}.h5"
    if not fpath.exists():
        raise FileNotFoundError(f"Processed halo HDF5 not found: {fpath}")

    out = {}
    with h5py.File(fpath, "r") as f:
        for key in ["N_halos", "M_halos", "pos_halos", "c_halos_sim", "v_halos_diff"]:
            out[key] = f[key][:]
    return out, fpath


def make_truth_sample_out(truth_targets):
    return {
        "ntot": [truth_targets["N_halos"].reshape(-1).astype(np.float32)],
        "m1": [truth_targets["M1_norm"].reshape(-1).astype(np.float32)],
        "mdiff": [truth_targets["Mdiff_norm"].reshape(-1, Nmax - 1).astype(np.float32)],
        "vel": [truth_targets["v_norm"].reshape(-1, Nmax * 3).astype(np.float32)],
        "conc": [truth_targets["c_norm"].reshape(-1, Nmax).astype(np.float32)],
        "pos": [truth_targets["pos_norm"].reshape(-1, Nmax * 3).astype(np.float32)],
    }


def make_truth_tensors(truth_targets):
    tensors = {
        "nhalos_truth": torch.from_numpy(truth_targets["N_halos"].reshape(1, total_vox, 1).astype(np.float32)),
        "m1_truth": torch.from_numpy(truth_targets["M1_norm"].reshape(1, total_vox, 1).astype(np.float32)),
        "mhalos_truth": torch.from_numpy(truth_targets["M_norm"].reshape(1, total_vox, Nmax).astype(np.float32)),
        "mdiff_truth": torch.from_numpy(truth_targets["Mdiff_norm"].reshape(1, total_vox, Nmax - 1).astype(np.float32)),
        "pos_truth": torch.from_numpy(truth_targets["pos_norm"].reshape(1, total_vox, Nmax * 3).astype(np.float32)),
    }
    return {key: value.to(DEVICE) for key, value in tensors.items()}


print("Loading training-format truth HDF5 and rebuilding normalized targets...")
halos_full, halo_h5_path = load_processed_halo_h5()
halo_sub = {key: subvols_unpadded(value, nb, nax) for key, value in halos_full.items()}

cosmo_sub = np.broadcast_to(
    cosmo_vals[None, None, None, None, :],
    (nsubs, nax, nax, nax, len(cosmo_vals)),
).copy()

truth_targets = prep_halo_catalog(
    df_Mh=halo_sub["M_halos"].astype(np.float32),
    df_Nh=halo_sub["N_halos"].astype(np.int16),
    cosmo=cosmo_sub.astype(np.float32),
    Mmin=lgMmin,
    Mmax=lgMmax,
    Nmax=Nmax,
    rescale_sub=float(sc.get("rescale_sub", 0.0)),
    df_v=halo_sub["v_halos_diff"].astype(np.float32),
    df_c=halo_sub["c_halos_sim"].astype(np.float32),
    df_pos=halo_sub["pos_halos"].astype(np.float32),
    vmin=vmin,
    vmax=vmax,
    cmin=cmin,
    cmax=cmax,
)

sample_out_truth = make_truth_sample_out(truth_targets)
truth_tensors = make_truth_tensors(truth_targets)
print(f"truth source HDF5: {halo_h5_path}")
print(f"truth occupied voxels: {(truth_targets['N_halos'].reshape(-1) > 0).sum():,}")
print(f"truth total halos: {truth_targets['N_halos'].reshape(-1).sum():,}")

In [ ]:
def run_model_sample(label, seed, **sample_kwargs):
    print(f"Running rollout: {label}")
    torch.manual_seed(seed)
    if DEVICE.startswith("cuda"):
        torch.cuda.manual_seed_all(seed)
    with torch.no_grad():
        return model.sample(
            cond_x,
            cond_x_nsh,
            cond_cosmo,
            **truth_tensors,
            binary_target_prior=binary_target_prior,
            **sample_kwargs,
        )


rollouts = {
    "truth_all": sample_out_truth,
}

rollouts["free"] = run_model_sample(
    "free",
    SEED,
    sample_binary=True,
    sample_multi=True,
    sample_m1=True,
    sample_mdiff=True,
    sample_vel=True,
    sample_conc=True,
    sample_pos=True,
    use_truth_masses=False,
)

rollouts["true_count_sampled_downstream"] = run_model_sample(
    "true_count_sampled_downstream",
    SEED + 1,
    sample_binary=False,
    sample_multi=False,
    sample_m1=True,
    sample_mdiff=True,
    sample_vel=True,
    sample_conc=True,
    sample_pos=True,
    use_truth_masses=False,
)

rollouts["true_count_truth_mass_sampled_props"] = run_model_sample(
    "true_count_truth_mass_sampled_props",
    SEED + 2,
    sample_binary=False,
    sample_multi=False,
    sample_m1=False,
    sample_mdiff=False,
    sample_vel=True,
    sample_conc=True,
    sample_pos=True,
    use_truth_masses=True,
)

rollouts["sampled_count_truth_overlap"] = hybrid_sampled_count_truth_overlap(
    rollouts["free"], rollouts["truth_all"]
)

print("Done.")

In [ ]:
diag = {name: rollout_diagnostics(sample_out) for name, sample_out in rollouts.items()}
truth_diag = diag["truth_all"]

rows = []
for name, d in diag.items():
    rows.append({
        "rollout": name,
        "N_occ": d["N_occ"],
        "N_occ_ratio": d["N_occ"] / max(truth_diag["N_occ"], 1),
        "N_halo": d["N_halo"],
        "N_halo_ratio": d["N_halo"] / max(truth_diag["N_halo"], 1),
        "mean_N_occupied": d["mean_N_occupied"],
        "mass_mono_bad_frac": d["mass_mono_bad_frac"],
        "nan_inf_frac": d["nan_inf_frac"],
    })

show_table(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
x = np.arange(Nmax + 1)
truth_pdf = np.maximum(diag["truth_all"]["Ntot_pdf"].astype(np.float64), 1.0)

for name, d in diag.items():
    if name == "truth_all":
        continue
    axes[0].plot(x, d["Ntot_pdf"], marker="o", label=name)
    axes[1].plot(x, d["Ntot_pdf"] / truth_pdf, marker="o", label=name)

axes[0].plot(x, diag["truth_all"]["Ntot_pdf"], color="black", marker="o", lw=2, label="truth_all")
axes[0].set_yscale("log")
axes[0].set_xlabel("N halos per voxel")
axes[0].set_ylabel("voxel count")
axes[0].legend(frameon=False, fontsize=8)

axes[1].axhline(1.0, color="black", ls="--", lw=1)
axes[1].set_xlabel("N halos per voxel")
axes[1].set_ylabel("ratio to truth_all")
axes[1].set_ylim(0.0, 2.0)
axes[1].legend(frameon=False, fontsize=8)
plt.show()

In [ ]:
print("Reconstructing subvolume-local catalogs...")
catalogs = {
    name: reconstruct_sample_out_by_subvolume(sample_out, vel_interps)
    for name, sample_out in tqdm(rollouts.items())
}
full_catalogs = {name: merge_subvolume_catalogs(cats) for name, cats in catalogs.items()}

for name, cat in full_catalogs.items():
    print(f"{name:36s}: {len(cat['pos']):,} halos")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

m_bins, hmf_truth = halo_mass_function(
    full_catalogs["truth_all"]["lgM"], lgMmin, lgMmax, n_bins=28, BoxSize=BoxSize
)
hmf_truth_safe = np.maximum(hmf_truth, 1e-30)

axes[0].plot(m_bins, hmf_truth, color="black", lw=2, label="truth_all")
for name, cat in full_catalogs.items():
    if name == "truth_all":
        continue
    _, hmf = halo_mass_function(cat["lgM"], lgMmin, lgMmax, n_bins=28, BoxSize=BoxSize)
    axes[0].plot(m_bins, hmf, lw=1.4, label=name)
    axes[1].plot(m_bins, hmf / hmf_truth_safe, lw=1.4, label=name)

axes[0].set_yscale("log")
axes[0].set_xlabel("log10 M")
axes[0].set_ylabel("dN/dlogM / volume")
axes[0].legend(frameon=False, fontsize=8)

axes[1].axhline(1.0, color="black", ls="--", lw=1)
axes[1].set_xlabel("log10 M")
axes[1].set_ylabel("HMF ratio to truth_all")
axes[1].set_ylim(0.0, 2.0)
axes[1].legend(frameon=False, fontsize=8)
plt.show()

In [ ]:
def selected_subvolume_ids():
    ids = np.arange(nsubs, dtype=np.int64)
    if MAX_POWER_SUBVOLS is not None:
        ids = ids[: int(MAX_POWER_SUBVOLS)]
    return ids


def catalog_for_power(cat, stat_key):
    pos = cat["pos"]
    weights = None
    if "rsd" in stat_key:
        pos = apply_rsd(pos, cat["vel"], Lsub, redshift, cosmo_vals, axis=RSD_AXIS)
    if "mass" in stat_key:
        weights = (10.0 ** cat["lgM"] / 1.0e14).astype(np.float32)
    return pos, weights


def compute_power_table(cats, stat_key):
    rows = []
    for sid in tqdm(selected_subvolume_ids(), desc=stat_key):
        cat = cats[int(sid)]
        if len(cat["pos"]) < 2:
            rows.append((None, None))
            continue
        pos, weights = catalog_for_power(cat, stat_key)
        k, pk = power_spectrum(pos, weights, POWER_GRID, Lsub, kmax=K_MAX, MAS=MAS, axis=RSD_AXIS)
        rows.append((k.astype(np.float64), pk.astype(np.float64)))
    return rows


def ratio_from_tables(num_table, den_table, k_ref):
    ratios = []
    for (k_num, p_num), (k_den, p_den) in zip(num_table, den_table):
        if k_num is None or k_den is None or len(k_num) < 2 or len(k_den) < 2:
            ratios.append(np.full_like(k_ref, np.nan, dtype=np.float64))
            continue
        p_num_i = np.interp(k_ref, k_num, p_num, left=np.nan, right=np.nan)
        p_den_i = np.interp(k_ref, k_den, p_den, left=np.nan, right=np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = p_num_i / p_den_i
        ratio[~np.isfinite(ratio)] = np.nan
        ratios.append(ratio)
    return np.vstack(ratios)


stat_specs = [
    ("real_number", "real-space number P(k)"),
    ("real_mass", "real-space mass-weighted P(k)"),
    ("rsd_number", "RSD number P(k)"),
    ("rsd_mass", "RSD mass-weighted P(k)"),
]

In [ ]:
if RUN_POWER_SPECTRA:
    stat_tables = {}
    for stat_key, _ in stat_specs:
        stat_tables[stat_key] = {
            name: compute_power_table(cats, stat_key)
            for name, cats in catalogs.items()
        }

    ratio_results = {}
    for stat_key, _ in stat_specs:
        den_table = stat_tables[stat_key]["truth_all"]
        k_ref = None
        for k, pk in den_table:
            if k is not None and len(k) > 2:
                k_ref = k.astype(np.float64)
                break
        if k_ref is None:
            raise RuntimeError(f"No valid truth_all power spectra for {stat_key}.")

        ratio_results[stat_key] = {"k": k_ref}
        for name in catalogs:
            if name == "truth_all":
                continue
            ratio_results[stat_key][name] = ratio_from_tables(
                stat_tables[stat_key][name], den_table, k_ref
            )
else:
    ratio_results = {}

In [ ]:
def draw_rollout_ratio_panel(ax, k, ratio_by_name, title):
    for name, ratios in ratio_by_name.items():
        mean = np.nanmean(ratios, axis=0)
        p16, p84 = np.nanpercentile(ratios, [16, 84], axis=0)
        line = ax.plot(k, mean, lw=1.6, label=name)[0]
        ax.fill_between(k, p16, p84, color=line.get_color(), alpha=0.12, lw=0)

    ax.axhline(1.0, color="black", ls="--", lw=1.0)
    ax.axhline(0.9, color="0.55", ls="--", lw=0.8)
    ax.axhline(1.1, color="0.55", ls="--", lw=0.8)
    ax.set_xscale("log")
    ax.set_ylim(*RATIO_YLIM)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("ratio to truth_all")


if RUN_POWER_SPECTRA:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, constrained_layout=True)
    axes = axes.ravel()

    for ax, (stat_key, stat_label) in zip(axes, stat_specs):
        k = ratio_results[stat_key]["k"]
        ratio_by_name = {
            name: ratios
            for name, ratios in ratio_results[stat_key].items()
            if name != "k"
        }
        draw_rollout_ratio_panel(ax, k, ratio_by_name, stat_label)

    for ax in axes[-2:]:
        ax.set_xlabel("k [h/Mpc]")
    axes[0].legend(frameon=False, fontsize=8)

    fig.suptitle(
        f"CHARM oracle rollout power ratios, sim {SIM_ID}, "
        f"{len(selected_subvolume_ids())} subvolumes, Lsub={Lsub:.1f} Mpc/h",
        fontsize=12,
    )
    FIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_PATH, bbox_inches="tight")
    plt.show()
    print(f"Saved figure to {FIG_PATH}")